In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df=pd.read_csv("D:/DIYA/PROJECTS/customer-churn-xai/data/Churn.csv")
print(f"Dataframe shape: {df.shape}")
print(df.info())

Dataframe shape: (7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 no

In [8]:
#dropping non-predictive columns
df.drop(columns=["customerID"], inplace=True, errors="ignore")

#coercing whitespace strings to NaN, then imputing with 0.0, i.e, accounts with 0 tenure
if df["TotalCharges"].dtype == "object":
    df["TotalCharges"]=pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
df["TotalCharges"]=df["TotalCharges"].fillna(0.0)
print(f"Missing values per column:\n {df.isnull().sum()[df.isnull().sum()>0]}")

Missing values per column:
 Series([], dtype: int64)


In [13]:
#grouping tenure into cohorts
tenure_bins=[-1, 12, 24, 48, np.inf]
tenure_labels=["0-12 months","13-24 months", "25-48months", "49+ months"]
df["TenureGroup"]=pd.cut(df["tenure"], bins=tenure_bins, labels=tenure_labels)

#total active add-on servives count
services=["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
df["ServiceCount"]=df[services].apply(lambda row: sum(row=="Yes"), axis=1)

#monthy spend ratio
df["AvgMonthlySpend"]=np.where(
    df["tenure"]>0,
    df["TotalCharges"]/df["tenure"],
    df["MonthlyCharges"]
)

df[["tenure", "TenureGroup","ServiceCount", "MonthlyCharges", "AvgMonthlySpend"]].head()

,tenure,TenureGroup,ServiceCount,MonthlyCharges,AvgMonthlySpend
0,1,0-12 months,1,29.85,29.850000
1,34,25-48months,3,56.95,55.573529
2,2,0-12 months,3,53.85,54.075000
3,45,25-48months,3,42.30,40.905556
4,2,0-12 months,1,70.70,75.825000


In [14]:
#target mapping and one-hot encoding

#map target binary integers
df["Churn"]=df["Churn"].map({"Yes": 1, "No": 0})

#seperate features from target
X=df.drop(columns=["Churn"])
Y=df["Churn"]

#one-hot encode remaining categorical features
X_encoded=pd.get_dummies(X, drop_first=True)

print(f"Encoded feature shape: {X_encoded.shape}")
print(f"Target Distribution:\n{Y.value_counts(normalize=True)}")

Encoded feature shape: (7043, 35)
Target Distribution:
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


In [15]:
#80/20 stratified train-test split and export

# Stratified partition preserving the ~26.5% minority churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, Y, test_size=0.20, random_state=42, stratify=Y
)

# Export processed matrices to data/
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

print("\n--- Phase 1 Final Artifacts ---")
print(f"X_train: {X_train.shape} | Churn Rate: {y_train.mean():.4f}")
print(f"X_test:  {X_test.shape}  | Churn Rate: {y_test.mean():.4f}")


--- Phase 1 Final Artifacts ---
X_train: (5634, 35) | Churn Rate: 0.2654
X_test:  (1409, 35)  | Churn Rate: 0.2654
